# 08 — Máquinas de Vetores de Suporte

> **Status:** notebook revisado e validado para estudo em JupyterLab.
>
> **Continuidade:** exemplos com dados sintéticos ou datasets integrados ao scikit-learn, sem acesso à internet.

**Objetivo geral:** construir compreensão conceitual e prática para aplicar este tema em problemas de Machine Learning, com avaliação adequada e interpretação dos resultados.


### Materiais complementares

- [Guia Técnico](./reference_materials/08_svm.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_svm.pdf)


## 1. Fundamentos, objetivos e contexto

**Objetivos.** Entender hiperplano de margem máxima, vetores de suporte, margem suave, kernels, $C$ e $\gamma$.

Uma SVM linear escolhe fronteira com margem ampla, tolerando violações em casos não separáveis. Kernel RBF permite fronteiras não lineares por produto interno implícito em espaço de características. SVM é sensível à escala e pode ser dispendiosa em datasets extensos.


## 2. Fundamentos matemáticos

Hiperplano $w^\top x+b=0$. Uma forma de otimização de margem suave é $\min_{w,b,\xi}\frac12\lVert w\rVert^2+C\sum_i\xi_i$ sujeito a $y_i(w^\top x_i+b)\ge1-\xi_i$ e $\xi_i\ge0$. Kernel RBF: $K(x,z)=\exp(-\gamma\lVert x-z\rVert_2^2)$.

**Conexão com disciplinas anteriores:** revise os conceitos matemáticos e estatísticos correspondentes. Não repetimos integralmente os capítulos de Python, visualização e inferência estatística.


## 3. Experimento guiado: preparando a pergunta

**Antes do código**, identifique: (a) quais dados são de treino, (b) qual informação poderá entrar no modelo, (c) que métrica ou propriedade será observada, e (d) que conclusão não é permitida pelos resultados. Estes exemplos são didáticos, com semente fixa onde aplicável.


In [1]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
X,y=make_moons(n_samples=300,noise=.20,random_state=42)
a,b,c,d=train_test_split(X,y,stratify=y,test_size=.25,random_state=42)
for kernel in ['linear','rbf']:
    model=make_pipeline(StandardScaler(),SVC(kernel=kernel,C=1.,gamma='scale')).fit(a,c)
    print(kernel,'treino',round(model.score(a,c),3),'teste',round(model.score(b,d),3),'vetores',model.named_steps['svc'].n_support_)

linear treino 0.884 teste 0.853 vetores [34 35]
rbf treino 0.942 teste 0.907 vetores [31 31]


### Como interpretar

Os números são resultados específicos do experimento e da amostra, não garantias universais. Verifique o que mudou, qual método foi usado e qual é o cenário de aplicação. Compare com a teoria matemática e não confunda desempenho de treino, validação e teste.


## 4. Segundo experimento ou extensão controlada

Mude apenas uma condição principal (algoritmo, parâmetro ou tipo de dado), quando possível, e registre o efeito. Não ajuste decisões com base no teste final.


In [2]:
from sklearn.model_selection import StratifiedKFold
cv=StratifiedKFold(4,shuffle=True,random_state=42)
for C in [.1,1,10]:
    p=make_pipeline(StandardScaler(),SVC(kernel='rbf',C=C,gamma='scale'))
    print('C:',C,'CV:',round(cross_val_score(p,a,c,cv=cv).mean(),3))

C: 0.1 CV: 0.889
C: 1 CV: 0.925
C: 10 CV: 0.947


## 5. Exercício resolvido

**Enunciado:** reforce o conceito central com um exemplo menor, cujo resultado pode ser inspecionado diretamente.


In [3]:
# Exercício resolvido: calcule um kernel RBF manualmente
import numpy as np
x=np.array([0.,1.]);z=np.array([2.,2.]);gamma=.5
print('K_RBF:',round(float(np.exp(-gamma*np.sum((x-z)**2))),4))

K_RBF: 0.0821


## 6. Interpretação, limitações e erros comuns

`C` alto penaliza violações; não garante melhor generalização. `gamma` alto restringe influência local no RBF, podendo sobreajustar. Para regressão com margem $\epsilon$, veja o capítulo 13 sobre SVR.

**Perguntas de revisão:**

1. Qual hipótese implícita sustenta o procedimento usado?
2. Que erro de implementação ou de validação poderia tornar o resultado artificialmente otimista?
3. Como você mudaria o experimento para refletir dados reais?


## 7. Consulta rápida

- SVC: classificação; SVR: regressão.
- Usar `StandardScaler` em `Pipeline`.
- Kernels: linear e RBF são pontos de partida.
- Escolher `C`/`gamma` por validação, não pelo teste.


## 8. Exercícios adicionais (para fazer no JupyterLab)

**Exercício A — sensibilidade:** altere um hiperparâmetro/hipótese e apresente uma tabela pequena com resultados e explicação. **Exercício B — revisão crítica:** construa um exemplo de uso incorreto (sem aplicá-lo a dados reais) e explique o procedimento correto. **Exercício C — comunicação:** escreva um parágrafo sobre escopo, métricas, suposições e limitações do método.

**Critério de aprovação:** códigos executam do começo ao fim, não há seleção pelo teste final e cada resultado possui interpretação.


## 9. Fontes e continuidade curricular

**Origem principal das aulas:** 12-svm.pdf; Support_Vector_Regression.ipynb.

**Referências de atualização:** documentação de `scikit-learn` (<https://scikit-learn.org/stable/>); explicações matemáticas redigidas para esta reconstrução. Outros pacotes (Optuna, XGBoost, SHAP, imbalanced-learn) não são necessários para executar este capítulo.



## Aprofundamento v2 — auditoria e aplicação

**Ponto de atenção deste capítulo:** margem e escala. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **margem e escala** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **margem e escala** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.
